# Ibis: Bringing Optionality to Python Dataframes
### Live demo notebook

This notebook is the live-coding companion to the talk. Run the cells top to
bottom during the session, or work through it on your own afterward — it's
fully self-contained for the DuckDB part, and clearly marked where you'd
point at a real Trino cluster instead.

**What you'll see:**
1. Connect Ibis to DuckDB and load a small example dataset
2. Build a lazy dataframe expression and execute it
3. Peek at the SQL Ibis generates under the hood
4. Mix raw SQL with the dataframe API
5. **The main event:** run the *exact same* analysis code against a Trino
   cluster by changing only the connection

See `SETUP.md` in this folder for install instructions and options for
getting access to a Trino endpoint (including a no-install hosted option).


## 1. Connect and load data

We start on DuckDB — it runs in-process, needs no server, and is the
default backend for local Ibis development. This is the "iterate locally"
half of Ibis's pitch.

**Presenting live?** Don't rely on venue wifi mid-talk. Run
`python prep_offline_data.py` once beforehand (see `SETUP.md`) — it downloads
the dataset and caches it as `data/penguins.parquet`. The cell below loads
from that local file if it exists, and only reaches out to the network as
a fallback.


In [ ]:
import pathlib
import ibis

# Interactive mode: expressions print their results immediately, like a
# pandas REPL session. Great for live demos; turn this off in production
# code / scripts, where you want execute() to be an explicit, deliberate step.
ibis.options.interactive = True

con = ibis.duckdb.connect()
con

In [ ]:
DATA_PATH = pathlib.Path("data/penguins.parquet")

if DATA_PATH.exists():
    # Cached locally by prep_offline_data.py -- no network needed on stage.
    penguins = con.read_parquet(DATA_PATH, table_name="penguins")
else:
    # Fallback: fetches from the network. `ibis.examples` ships a handful of
    # small, well-known datasets for exactly this kind of demo -- `penguins`
    # is the dataframe-tutorial equivalent of iris.
    penguins = ibis.examples.penguins.fetch(backend=con)

penguins.head()

## 2. Build a lazy expression

Every call below (`filter`, `group_by`, `aggregate`, `order_by`) just adds
a node to an expression tree — **no query has run yet**. Nothing touches
DuckDB until we ask for a result.

Notice this is also written as a plain function. That's deliberate: this
function doesn't know or care what backend `t` came from. That's the whole
trick we'll lean on in Part 4.


In [ ]:
def top_species_by_mass(t):
    """Average body mass per species, heaviest first.

    Works on a table from ANY Ibis backend -- DuckDB, Trino, Snowflake,
    Polars, BigQuery... the function itself never changes.
    """
    return (
        t.filter(t.body_mass_g.notnull())
         .group_by("species")
         .aggregate(
             avg_mass_g=t.body_mass_g.mean(),
             n=t.count(),
         )
         .order_by(ibis.desc("avg_mass_g"))
    )

result = top_species_by_mass(penguins)
result

> Because `ibis.options.interactive = True`, Ibis quietly called `.execute()`
> for us above to render that table. In a script, you'd call it explicitly:
> `result.execute()` &rarr; returns a pandas DataFrame (or `.to_polars()`,
> `.to_pyarrow()`, `.to_pandas()`, etc., if you want a different in-memory
> format).


## 3. Peek under the hood: it's just SQL

Most Ibis backends work by compiling your expression into that backend's
native SQL dialect. You never *have* to look at this, but it's reassuring
(and a good debugging tool) to know it's there.


In [ ]:
print(ibis.to_sql(result))

## 4. Mixing raw SQL with the dataframe API

Sometimes SQL is just the clearest way to say what you mean — a gnarly
window function, or a query someone already handed you. Ibis lets you drop
into SQL and pick the dataframe API back up on the result, in either
direction.


In [ ]:
# Backend.sql(...) runs a raw SQL string and hands back a normal, chainable
# Ibis table expression.
by_island = con.sql("""
    SELECT species, island, count(*) AS n
    FROM penguins
    GROUP BY 1, 2
""")

# ...and now we're back in the dataframe API, on the SQL query's result.
(
    by_island
    .mutate(pct_of_total=by_island.n / by_island.n.sum())
    .order_by(ibis.desc("n"))
)

## 5. The main event: swap DuckDB for Trino

This is the "optionality" in the talk title. Below, the **only** thing that
changes is how we open the connection and grab the table. The analysis
function from Part 2 — `top_species_by_mass` — is copy-pasted, unchanged.

**Before running this section:** point `TRINO_*` at a real endpoint. See
`SETUP.md` for two ways to get one in about five minutes (a local
single-node Trino via Docker, or a free hosted Starburst Galaxy cluster —
the same managed Trino the official Ibis Trino tutorial uses).


In [ ]:
import os

# Fill these in, or export them as environment variables before starting
# Jupyter so they're never typed into the notebook itself.
TRINO_HOST = os.environ.get("TRINO_HOST", "localhost")
TRINO_PORT = int(os.environ.get("TRINO_PORT", 8080))
TRINO_USER = os.environ.get("TRINO_USER", "demo")
TRINO_CATALOG = os.environ.get("TRINO_CATALOG", "memory")   # aka "database" in ibis.trino.connect
TRINO_SCHEMA = os.environ.get("TRINO_SCHEMA", "default")
TRINO_PASSWORD = os.environ.get("TRINO_PASSWORD")  # only needed for Starburst Galaxy / secured clusters

In [ ]:
# --- THE ONE LINE THAT CHANGES -----------------------------------------
trino_con = ibis.trino.connect(
    host=TRINO_HOST,
    port=TRINO_PORT,
    user=TRINO_USER,
    password=TRINO_PASSWORD,
    database=TRINO_CATALOG,
    schema=TRINO_SCHEMA,
)
# -------------------------------------------------------------------------

# Load the same dataset into the Trino-visible catalog so there's something
# to query. On a real company cluster, this step doesn't exist -- the table
# already lives in the warehouse. We only do it here because we're demoing
# against a scratch cluster with nothing in it yet.
trino_con.create_table("penguins", penguins.to_pandas(), overwrite=True)

penguins_on_trino = trino_con.table("penguins")

# Same function. Same code. Different engine.
top_species_by_mass(penguins_on_trino)

That query just ran as a distributed SQL query on Trino instead of an
in-process DuckDB query — and `top_species_by_mass` never found out.
Point the same function at Snowflake, BigQuery, Polars, or PySpark and
nothing about it needs to change either.

**That's the pitch:** prototype locally, deploy to whatever your team
actually runs in production, and keep one dataframe API the whole way.


## Try it yourself

- Swap `top_species_by_mass` for your own analysis and re-run it against
  both backends.
- Try `ibis.polars.connect()` as a third backend for the same function.
- Read the generated SQL for the Trino run too:
  `print(ibis.to_sql(top_species_by_mass(penguins_on_trino)))`
- Official docs & tutorials: <https://ibis-project.org>
- This notebook + setup instructions: see `SETUP.md` in this folder
